# Task 2 — Fairness Algorithm Evaluation

Serena van Leeuwen
7102380

**Selected paper:** Zafar et al. (2017), *Fairness Constraints: Mechanisms for Fair Classification*.



## workflow
1. **Before training or mitigation:** calculate DP_Diff, DIR and consistency from each dataset's original labels.
2. **Split:** create a stratified train/test split.
3. **Apply the selected method:** train the paper's in-processing fairness-constrained logistic-regression method on the training set.
4. **Held-out evaluation:** calculate DP_Diff, DIR, consistency and predictive metrics on the test set using the fair model's predictions.
5. **Compare:** place the original-label measurements beside the fair model's held-out test measurements.


## How the paper's method is represented
Zafar et al. constrain the covariance between the sensitive attribute and the signed decision-boundary score. This notebook uses logistic loss and constrains the absolute covariance to be at most `COVARIANCE_THRESHOLD`. It is a Python 3 reimplementation of the paper's logistic-regression formulation using SciPy SLSQP.

The sensitive attribute is excluded from predictor features. It is supplied separately during training for the fairness constraint and during evaluation for group fairness metrics.


## 1. Install and import packages

In [34]:
#%pip -q install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import expit
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 123
TEST_SIZE = 0.30
K_CONSISTENCY = 5
COVARIANCE_THRESHOLD = 0.0

## 2. Fairness and predictive metrics

- **DP_Diff:** positive-outcome rate in the protected group minus the rate in the comparison group.
- **DIR:** unprivileged-group positive-outcome rate divided by the comparison-group rate. This is directional, so it can exceed 1.
- **Consistency:** k-nearest-neighbour consistency (`k=5`), measuring how often nearby feature vectors receive the same label or prediction.
- **Accuracy and F1:** predictive metrics reported for the fair classifier on the held-out test set.

The symmetric p%-rule is included as an additional descriptive measure. It is not a substitute for directional DIR.


In [2]:
def group_rates(y_pred, z):
    # z=1 is the unprivileged group; z=0 is the comparison group.
    y_pred = np.asarray(y_pred).astype(int)
    z = np.asarray(z).astype(int)
    rate_unprivileged = y_pred[z == 1].mean()
    rate_comparison = y_pred[z == 0].mean()
    return rate_unprivileged, rate_comparison

def demographic_parity_difference(y_pred, z):
    # Signed difference: unprivileged minus comparison group.
    rp, rc = group_rates(y_pred, z)
    return rp - rc

def disparate_impact_ratio(y_pred, z):
    # Directional DIR = unprivileged selection rate / comparison selection rate.
    rp, rc = group_rates(y_pred, z)
    if rc == 0:
        return np.inf if rp > 0 else np.nan
    return rp / rc

def p_rule(y_pred, z):
    # Symmetric version corresponding to the paper's p%-rule, expressed as a ratio.
    rp, rc = group_rates(y_pred, z)
    if max(rp, rc) == 0:
        return np.nan
    return min(rp, rc) / max(rp, rc)

def consistency_score(X, labels, k=5, max_rows=5000, random_state=42):
    # k-NN consistency: 1 - average disagreement between each label and its neighbours.
    # A subset is used for speed on larger datasets.
    X = np.asarray(X)
    labels = np.asarray(labels).astype(int)
    if len(labels) > max_rows:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(len(labels), size=max_rows, replace=False)
        X, labels = X[idx], labels[idx]
    k_eff = min(k + 1, len(labels))
    nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean")
    nn.fit(X)
    neighbours = nn.kneighbors(X, return_distance=False)[:, 1:]
    disagreement = np.abs(labels[:, None] - labels[neighbours]).mean()
    return 1.0 - disagreement

def metric_row(y_true, y_pred, z, X_for_consistency, label):
    return {
        "Result": label,
        "DP_Diff (unprivileged - comparison)": demographic_parity_difference(y_pred, z),
        "DIR (unprivileged / comparison)": disparate_impact_ratio(y_pred, z),
        "p%-rule (symmetric)": p_rule(y_pred, z),
        "Consistency": consistency_score(X_for_consistency, y_pred, k=K_CONSISTENCY),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

## 3. Paper method: in-processing fairness-constrained logistic regression

Zafar et al. (2017) propose constraining the covariance between the binary sensitive attribute `z` and the signed score/distance to the classifier's decision boundary. For a linear logistic classifier, this notebook imposes:

The model minimizes logistic loss subject to this covariance constraint. The sensitive attribute is used in the training constraint but is not included among the model's prediction features.

The constraint is applied to the **training split**. Fairness metrics are measured on the **held-out test split**, so test-set covariance is not expected to be exactly zero.


In [3]:
def fit_zafar_logistic(X, y01, z, fair=True, cov_threshold=0.0, maxiter=5000):
    X = np.asarray(X, dtype=float)
    y01 = np.asarray(y01).astype(int)
    z = np.asarray(z).astype(float)

    y_pm = np.where(y01 == 1, 1.0, -1.0)
    n, p = X.shape

    # Mean logistic loss; a tiny ridge term stabilises optimisation for correlated one-hot features.
    ridge = 1e-7
    def objective(w):
        margins = y_pm * (X @ w)
        return np.mean(np.logaddexp(0.0, -margins)) + 0.5 * ridge * np.dot(w, w)

    def gradient(w):
        margins = y_pm * (X @ w)
        coeff = -y_pm * expit(-margins)
        return (X.T @ coeff) / n + ridge * w

    constraints = []
    if fair:
        z_centered = z - z.mean()
        cov_vector = (X.T @ z_centered) / n
        # SLSQP inequality constraints must be >= 0.
        constraints = [
            {"type": "ineq", "fun": lambda w: cov_threshold - np.dot(cov_vector, w),
             "jac": lambda w: -cov_vector},
            {"type": "ineq", "fun": lambda w: cov_threshold + np.dot(cov_vector, w),
             "jac": lambda w: cov_vector}
        ]

    result = minimize(
        objective,
        x0=np.zeros(p),
        jac=gradient,
        method="SLSQP",
        constraints=constraints,
        options={"maxiter": maxiter, "ftol": 1e-9, "disp": False}
    )

    if not result.success:
        print("Optimisation warning:", result.message)
    if fair:
        achieved_cov = float(np.dot((z - z.mean()), X @ result.x) / n)
        print(f"Training covariance = {achieved_cov:.6f} (allowed absolute value <= {cov_threshold})")
    return result.x

def predict_zafar(X, w):
    # The authors classify by the sign of the decision-boundary score.
    return (np.asarray(X) @ w >= 0).astype(int)

## 4. Dataset loaders

The loaders below create a common format: a feature DataFrame, binary target `y` (1 = positive outcome), and binary sensitive attribute `z` (1 = unprivileged group). The sensitive attribute is excluded from model predictors.

**Group definitions used in this notebook**
- Adult: women (`Female`) are unprivileged; positive outcome is income above 50K.
- German Credit: people younger than 25 or older than 60 are unprivileged; positive outcome is good credit (class 1).
- COMPAS: Black defendants are unprivileged; positive outcome is no two-year recidivism.

These are explicit analysis choices, not universal definitions of unprivileged groups. State them in your report.

In [4]:
def load_adult():
    adult = fetch_openml("adult", version=2, as_frame=True)
    df = adult.data.copy()
    target = adult.target.astype(str).str.strip().str.replace(".", "", regex=False)
    sex = df["sex"].astype(str).str.strip()
    y = (target == ">50K").astype(int)
    z = (sex == "Female").astype(int)  # unprivileged group = women
    X = df.drop(columns=["sex"]).copy()
    return X, y.to_numpy(), z.to_numpy()

def load_german():
    from ucimlrepo import fetch_ucirepo
    german = fetch_ucirepo(id=144)
    X = german.data.features.copy()
    target = german.data.targets.iloc[:, 0].astype(str).str.strip()

    # Class 1 = good credit, class 2 = bad credit
    y = (target == "1").astype(int).to_numpy()

    # Attribute13 represents age
    age_col = "Attribute13"
    if age_col not in X.columns:
        raise KeyError(
            f"Could not find {age_col}. "
            f"Available columns: {list(X.columns)}"
        )
    # Convert age to numeric
    age = pd.to_numeric(X[age_col], errors="coerce")

    # unprivileged group: younger than 25 or older than 60
    z = ((age < 25) | (age > 60)).astype(int).to_numpy()
    # Remove age from the model predictors
    X = X.drop(columns=[age_col])

    return X, y, z

def load_compas():
    url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
    df = pd.read_csv(url)

    # Common ProPublica two-year COMPAS sample filters.
    df = df[
        df["days_b_screening_arrest"].between(-30, 30)
        & (df["is_recid"] != -1)
        & (df["c_charge_degree"] != "O")
        & (df["score_text"] != "N/A")
        & (df["race"].isin(["African-American", "Caucasian"]))
    ].copy()

    y = (df["two_year_recid"] == 0).astype(int).to_numpy()  # positive = no recidivism
    z = (df["race"] == "African-American").astype(int).to_numpy()  # unprivileged = Black

    feature_cols = [
        "age", "sex", "age_cat", "juv_fel_count", "juv_misd_count",
        "juv_other_count", "priors_count", "c_charge_degree"
    ]
    X = df[feature_cols].copy()
    return X, y, z

loaders = {
    "Adult": load_adult,
    "German Credit": load_german,
    "COMPAS": load_compas
}

## 5. Task 2 experiment: original labels, stratified split, fair model, test evaluation

For each dataset, the code first measures the original labels, before fitting a classifier or applying mitigation. It then creates a stratified train/test split, one-hot encodes categorical predictors, trains the fairness-constrained classifier, and evaluates its predictions on the held-out test set.

Consistency is computed in a feature space where categorical predictors are one-hot encoded. No missing-value imputation or numerical scaling is applied since these methods were not specified in the paper. The one-hot encoder used for model training is fitted on the training data and then applied to the test data


In [5]:
def make_preprocessor(X_train):
    categorical = X_train.select_dtypes(include=["object", "str", "category", "bool"]).columns.tolist()
    numeric = [c for c in X_train.columns if c not in categorical]

    # No missing-value imputation or numerical scaling is applied.
    # One-hot encoding is retained only because logistic regression requires
    # numerical input features.
    return ColumnTransformer([
        ("num", "passthrough", numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ], remainder="drop")


def run_one_dataset(name, loader):
    print("\n" + "="*80)
    print(name)
    print("="*80)

    X_df, y, z = loader()
    X_df = X_df.reset_index(drop=True)
    y = np.asarray(y).astype(int)
    z = np.asarray(z).astype(int)

    valid = np.isfinite(y) & np.isfinite(z)
    X_df = X_df.loc[valid].reset_index(drop=True)
    y, z = y[valid], z[valid]

    if len(np.unique(y)) != 2:
        raise ValueError(f"{name}: target must contain both classes.")
    if len(np.unique(z)) != 2:
        raise ValueError(f"{name}: sensitive attribute must contain both groups.")

    print(f"Rows: {len(y):,}; features before encoding: {X_df.shape[1]}")
    print("unprivileged group proportion:", round(z.mean(), 3))
    print("Positive original-label proportion:", round(y.mean(), 3))

    # before training/mitigation: measure original labels.
    original_consistency_preprocessor = make_preprocessor(X_df)
    X_all_for_consistency = original_consistency_preprocessor.fit_transform(X_df)
    original_metrics = {
        "Dataset": name,
        "Stage": "Before mitigation: original labels",
        "DP_Diff": demographic_parity_difference(y, z),
        "DIR": disparate_impact_ratio(y, z),
        "p%-rule": p_rule(y, z),
        "Consistency": consistency_score(X_all_for_consistency, y, k=K_CONSISTENCY),
        "Accuracy": np.nan,
        "F1": np.nan
    }

    #stratified train/test split.
    X_train_df, X_test_df, y_train, y_test, z_train, z_test = train_test_split(
        X_df, y, z, test_size=TEST_SIZE, random_state=RANDOM_STATE,
        stratify=y
    )

    # Fit classifier preprocessing on training data only.
    model_preprocessor = make_preprocessor(X_train_df)
    X_train = model_preprocessor.fit_transform(X_train_df)
    X_test = model_preprocessor.transform(X_test_df)

    # Add intercept as in the paper's linear decision-boundary formulation.
    X_train_i = np.column_stack([np.ones(len(X_train)), X_train])
    X_test_i = np.column_stack([np.ones(len(X_test)), X_test])

    # TASK 2 — in-processing mitigation using Zafar et al.'s covariance constraint.
    w_fair = fit_zafar_logistic(
        X_train_i, y_train, z_train,
        fair=True, cov_threshold=COVARIANCE_THRESHOLD
    )
    pred_fair = predict_zafar(X_test_i, w_fair)

    #evaluate on held-out test data.
    fair_metrics = {
        "Dataset": name,
        "Stage": "After mitigation: fair model (held-out test)",
        "DP_Diff": demographic_parity_difference(pred_fair, z_test),
        "DIR": disparate_impact_ratio(pred_fair, z_test),
        "p%-rule": p_rule(pred_fair, z_test),
        "Consistency": consistency_score(X_test, pred_fair, k=K_CONSISTENCY),
        "Accuracy": accuracy_score(y_test, pred_fair),
        "F1": f1_score(y_test, pred_fair, zero_division=0)
    }

    result_df = pd.DataFrame([original_metrics, fair_metrics])
    display(result_df.round(4))
    return {
        "name": name, "metrics": result_df, "y_test": y_test,
        "z_test": z_test, "pred_fair": pred_fair, "weights_fair": w_fair
    }

## 6. Run Task 2 for all three datasets

The output for each dataset contains exactly the two requested stages: original-label fairness measurements before training/mitigation, and fairness plus predictive measurements for the fairness-constrained model on the held-out test set.


In [6]:
all_results = {}
for dataset_name, loader in loaders.items():
        all_results[dataset_name] = run_one_dataset(dataset_name, loader)
    


Adult
Rows: 48,842; features before encoding: 13
Protected group proportion: 0.332
Positive original-label proportion: 0.239
Optimisation warning: Positive directional derivative for linesearch
Training covariance = -0.000002 (allowed absolute value <= 0.0)


,Dataset,Stage,DP_Diff,DIR,p%-rule,Consistency,Accuracy,F1
0,Adult,Before mitigation: original labels,-0.1945,0.3597,0.3597,0.6928,NaN,NaN
1,Adult,After mitigation: fair model (held-out test),-0.0314,0.8194,0.8194,0.7860,0.8348,0.5897



German Credit
Rows: 1,000; features before encoding: 19
Protected group proportion: 0.194
Positive original-label proportion: 0.7
Training covariance = -0.000000 (allowed absolute value <= 0.0)


,Dataset,Stage,DP_Diff,DIR,p%-rule,Consistency,Accuracy,F1
0,German Credit,Before mitigation: original labels,-0.0819,0.8857,0.8857,0.6020,NaN,NaN
1,German Credit,After mitigation: fair model (held-out test),0.0456,1.0607,0.9428,0.6353,0.74,0.8219



COMPAS
Rows: 5,278; features before encoding: 8
Protected group proportion: 0.602
Positive original-label proportion: 0.53
Training covariance = -0.000000 (allowed absolute value <= 0.0)


,Dataset,Stage,DP_Diff,DIR,p%-rule,Consistency,Accuracy,F1
0,COMPAS,Before mitigation: original labels,-0.1323,0.7828,0.7828,0.6210,NaN,NaN
1,COMPAS,After mitigation: fair model (held-out test),0.0664,1.0963,0.9121,0.9355,0.5101,0.6104


## 7. Combined Task 2 results

The combined table presents the before-mitigation original-label measurements and the after-mitigation held-out test measurements for all three datasets.


In [7]:
combined = []
for name, result in all_results.items():
    tmp = result["metrics"].copy()
    tmp["Dataset"] = name
    combined.append(tmp)

combined_results = pd.concat(combined, ignore_index=True)
display(combined_results)

,Dataset,Stage,DP_Diff,DIR,p%-rule,Consistency,Accuracy,F1
0,Adult,Before mitigation: original labels,-0.194516,0.359655,0.359655,0.692800,NaN,NaN
1,Adult,After mitigation: fair model (held-out test),-0.031364,0.819428,0.819428,0.786000,0.834846,0.589691
2,German Credit,Before mitigation: original labels,-0.081860,0.885651,0.885651,0.602000,NaN,NaN
3,German Credit,After mitigation: fair model (held-out test),0.045573,1.060680,0.942792,0.635333,0.740000,0.821918
4,COMPAS,Before mitigation: original labels,-0.132279,0.782839,0.782839,0.621040,NaN,NaN
5,COMPAS,After mitigation: fair model (held-out test),0.066409,1.096344,0.912122,0.935480,0.510101,0.610442
